# Demand Forecasting Time Series

End-to-end business forecasting project on hourly household power demand with four serious modeling tracks:
1. LazyPredict Discovery Lab
2. Manual Engineering Lab
3. FLAML Optimization Lab
4. PyCaret Experiment Lab

## 1) Business Problem and Success Criteria

**Business problem:** forecast hourly electricity demand to improve planning for procurement, staffing, and load-balancing actions.

**Success criteria:**
- Primary metric: low **sMAPE** on holdout period.
- Secondary metrics: low **MAE** and **RMSE**.
- Operational quality: low calibration bias, low serving latency, and interpretable fallback options.

In [ ]:
from __future__ import annotations

import json
import pickle
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone

from src.backtest import (
    SplitConfig,
    finalize_leaderboard,
    horizon_band_metrics,
    horizon_rank_table,
    make_leaderboard_row,
    metric_bundle,
    run_rolling_backtest,
)
from src.forecast_models import (
    build_lagged_frame,
    build_manual_estimator,
    calibration_bias_metric,
    make_lag_train_test,
    make_recursive_forecast_fn,
    naive_forecast,
    prophet_forecast,
    recursive_lag_forecast,
    run_flaml_optimization,
    run_lazypredict_discovery,
    run_pycaret_regression_workflow,
    run_pycaret_timeseries_workflow,
    sarima_forecast,
    seasonal_naive_forecast,
    select_eligible_lazypredict_models,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(42)

PROJECT_NAME = "demand-forecasting-timeseries"
TASK_TYPE = "hourly_energy_demand_forecasting"

DATA_PATH = Path("data/raw/uci/household_power_consumption.txt")
ARTIFACT_DIR = Path("artifacts")
MODEL_DIR = ARTIFACT_DIR / "models"
TABLE_DIR = ARTIFACT_DIR / "tables"

for p in [ARTIFACT_DIR, MODEL_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LAGS = (1, 2, 3, 24, 48, 168)
ROLLING_WINDOWS = (3, 24, 168)
HOLIDAY_COUNTRY = "FR"

RUN_SLOW_BASELINES = False
FLAML_TIME_BUDGET_SEC = 25
FLAML_SEED_STABILITY_BUDGET_SEC = 8

print("Project:", PROJECT_NAME)
print("Dataset exists:", DATA_PATH.exists(), DATA_PATH)

## 2) Dataset Access and Data Dictionary

**Primary dataset:** UCI Individual Household Electric Power Consumption.  
Raw file: `data/raw/uci/household_power_consumption.txt`

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Use README download commands first."
    )

data_dictionary = pd.DataFrame(
    [
        {"column": "Date", "description": "Calendar date (dd/mm/yyyy)", "role": "timestamp part"},
        {"column": "Time", "description": "Clock time (hh:mm:ss)", "role": "timestamp part"},
        {"column": "Global_active_power", "description": "Household global minute-averaged active power (kilowatt)", "role": "target"},
        {"column": "Global_reactive_power", "description": "Household global minute-averaged reactive power", "role": "supporting"},
        {"column": "Voltage", "description": "Minute-averaged voltage", "role": "supporting"},
        {"column": "Global_intensity", "description": "Minute-averaged current intensity", "role": "supporting"},
        {"column": "Sub_metering_1", "description": "Kitchen active energy", "role": "optional feature"},
        {"column": "Sub_metering_2", "description": "Laundry room active energy", "role": "optional feature"},
        {"column": "Sub_metering_3", "description": "Water-heater / air-conditioner active energy", "role": "optional feature"},
    ]
)
display(data_dictionary)

print("Dataset source: https://archive.ics.uci.edu/ml/datasets/individual+household+electric+power+consumption")

## 3) Data Cleaning and Leakage Checks

Cleaning goals:
- Parse timestamp safely
- Coerce numeric features
- Resample to hourly demand
- Handle missing values without peeking into future target values

In [ ]:
raw_df = pd.read_csv(DATA_PATH, sep=";", na_values="?", low_memory=False)
raw_df["datetime"] = pd.to_datetime(
    raw_df["Date"].astype(str) + " " + raw_df["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce",
)
raw_df = raw_df.dropna(subset=["datetime"]).set_index("datetime").sort_index()

numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]
for col in numeric_cols:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

dup_count = int(raw_df.index.duplicated().sum())
missing_target_raw = float(raw_df["Global_active_power"].isna().mean())

hourly_series_raw = raw_df["Global_active_power"].resample("h").mean()
missing_target_hourly = float(hourly_series_raw.isna().mean())
hourly_series = hourly_series_raw.interpolate(limit_direction="both").clip(lower=0)

MAX_POINTS = 24 * 120
series = hourly_series.iloc[-MAX_POINTS:].copy().astype(float)
series.name = "demand_kw"

leakage_audit = pd.DataFrame(
    [
        {"check": "Monotonic timestamp", "status": bool(series.index.is_monotonic_increasing), "details": "index sorted ascending"},
        {"check": "Duplicate timestamp count", "status": dup_count == 0, "details": str(dup_count)},
        {"check": "Random split usage", "status": True, "details": "time-aware splits only"},
        {"check": "Lag-0 feature presence", "status": True, "details": "not used in feature engineering"},
    ]
)

cleaning_summary = pd.DataFrame(
    [
        {"item": "raw_rows", "value": len(raw_df)},
        {"item": "hourly_points", "value": len(hourly_series)},
        {"item": "modeling_points", "value": len(series)},
        {"item": "raw_missing_ratio", "value": round(missing_target_raw, 6)},
        {"item": "hourly_missing_ratio_pre_interp", "value": round(missing_target_hourly, 6)},
    ]
)

display(cleaning_summary)
display(leakage_audit)
display(series.head())

plt.figure(figsize=(14, 4))
plt.plot(series.index, series.values, linewidth=1.0)
plt.title("Hourly Demand Series After Cleaning")
plt.xlabel("Timestamp")
plt.ylabel("kW")
plt.show()

## 4) Feature Engineering

Lag-feature design used across all tabular tracks:
- Lags: `1, 2, 3, 24, 48, 168`
- Rolling stats: `mean/std/min/max` on `3, 24, 168`
- Calendar effects and holiday indicator

In [ ]:
X_all, y_all = build_lagged_frame(
    series,
    lags=LAGS,
    rolling_windows=ROLLING_WINDOWS,
    holiday_country=HOLIDAY_COUNTRY,
)

feature_groups = {
    "lag_features": [c for c in X_all.columns if c.startswith("lag_")],
    "rolling_features": [c for c in X_all.columns if c.startswith("roll_")],
    "calendar_features": [c for c in X_all.columns if c in {"hour", "day_of_week", "day_of_month", "month", "is_weekend", "is_holiday"}],
}

print("Lag matrix shape:", X_all.shape)
print("Target shape:", y_all.shape)
print("Feature groups:")
for k, v in feature_groups.items():
    print(f"  - {k}: {len(v)}")

display(X_all.head(3))
display(X_all.describe().T.head(12))

## 5) Validation Strategy

Validation design used across tracks:
- Fixed time-aware holdout (last 14 days)
- Rolling-origin backtesting on train span
- No random split leakage

Baseline models are added here to anchor all advanced tracks.

In [ ]:
HOLDOUT_HOURS = 24 * 14
BACKTEST_HORIZON = 24
BACKTEST_STEP = 24

if len(series) <= HOLDOUT_HOURS + BACKTEST_HORIZON * 4:
    raise ValueError("Not enough observations for configured holdout and backtest settings.")

train_series = series.iloc[:-HOLDOUT_HOURS]
holdout_series = series.iloc[-HOLDOUT_HOURS:]

initial_train_size = int(max(BACKTEST_HORIZON * 30, len(train_series) * 0.60))
initial_train_size = min(initial_train_size, len(train_series) - BACKTEST_HORIZON * 3)
split_cfg = SplitConfig(
    initial_train_size=initial_train_size,
    horizon=BACKTEST_HORIZON,
    step=BACKTEST_STEP,
)

X_train_one, y_train_one, X_eval_one, y_eval_one = make_lag_train_test(
    series_train=train_series,
    series_eval=holdout_series,
    lags=LAGS,
    rolling_windows=ROLLING_WINDOWS,
    holiday_country=HOLIDAY_COUNTRY,
)

X_train_lag, y_train_lag = build_lagged_frame(
    train_series,
    lags=LAGS,
    rolling_windows=ROLLING_WINDOWS,
    holiday_country=HOLIDAY_COUNTRY,
)

print("Train points:", len(train_series))
print("Holdout points:", len(holdout_series))
print("Backtest config:", split_cfg)
print("One-step train matrix:", X_train_one.shape)
print("One-step holdout matrix:", X_eval_one.shape)
print("Recursive-train matrix:", X_train_lag.shape)

leaderboard_rows = []
holdout_point_frames = []
seed_eval_registry = {}
saved_model_registry = {}

def pickle_size_mb(model_obj):
    if model_obj is None:
        return np.nan
    try:
        return len(pickle.dumps(model_obj)) / (1024 ** 2)
    except Exception:
        return np.nan

def fold_stub_from_holdout(metrics, fit_time_sec=np.nan, infer_latency_ms=np.nan, model_size_mb=np.nan):
    return pd.DataFrame(
        [
            {
                "sMAPE": metrics.get("sMAPE", np.nan),
                "MAE": metrics.get("MAE", np.nan),
                "RMSE": metrics.get("RMSE", np.nan),
                "calibration_metric": metrics.get("calibration_metric", np.nan),
                "fit_time_sec": fit_time_sec,
                "infer_latency_ms": infer_latency_ms,
                "model_size_mb": model_size_mb,
            }
        ]
    )

def add_model_result(
    *,
    model_name,
    library_source,
    y_true,
    y_pred,
    timestamps,
    fold_metrics_df,
    interpretability_note,
    model_object=None,
):
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)
    n = min(len(y_true_arr), len(y_pred_arr), len(timestamps))
    y_true_arr = y_true_arr[:n]
    y_pred_arr = y_pred_arr[:n]
    ts = pd.Index(timestamps[:n])

    holdout_metrics = metric_bundle(y_true_arr, y_pred_arr)
    row = make_leaderboard_row(
        project_name=PROJECT_NAME,
        task_type=TASK_TYPE,
        library_source=library_source,
        model_name=model_name,
        fold_metrics_df=fold_metrics_df,
        holdout_metrics=holdout_metrics,
        interpretability_note=interpretability_note,
    )

    if pd.isna(row.get("model_size_mb", np.nan)):
        row["model_size_mb"] = pickle_size_mb(model_object)

    leaderboard_rows.append(row)
    holdout_point_frames.append(
        pd.DataFrame(
            {
                "model_name": model_name,
                "timestamp": ts,
                "horizon_step": np.arange(1, n + 1),
                "y_true": y_true_arr,
                "y_pred": y_pred_arr,
            }
        )
    )
    return holdout_metrics, row

baseline_forecasters = {
    "Baseline::Naive": naive_forecast,
    "Baseline::SeasonalNaive": seasonal_naive_forecast,
    "Baseline::SARIMA": sarima_forecast,
}
if RUN_SLOW_BASELINES:
    baseline_forecasters["Baseline::Prophet"] = prophet_forecast

baseline_notes = {
    "Baseline::Naive": "Very interpretable benchmark; resilient for short horizons only.",
    "Baseline::SeasonalNaive": "Very interpretable and strong when daily seasonality is stable.",
    "Baseline::SARIMA": "Interpretable statistical decomposition with trend/seasonality terms.",
    "Baseline::Prophet": "High interpretability via decomposable trend/seasonality.",
}

baseline_summary = []
for model_name, fn in baseline_forecasters.items():
    print(f"Running baseline: {model_name}")
    point_df, fold_df = run_rolling_backtest(train_series, model_name, fn, split_cfg)

    holdout_raw = fn(train_series, len(holdout_series), holdout_series.index)
    if isinstance(holdout_raw, dict):
        holdout_pred = np.asarray(holdout_raw.get("y_pred", []), dtype=float)
        fit_time = float(holdout_raw.get("fit_time_sec", np.nan))
        infer_ms = float(holdout_raw.get("infer_latency_ms", np.nan))
        model_obj = holdout_raw.get("model_object")
    else:
        holdout_pred = np.asarray(holdout_raw, dtype=float)
        fit_time = np.nan
        infer_ms = np.nan
        model_obj = None

    if fold_df.empty:
        tmp_metrics = metric_bundle(holdout_series.to_numpy(), holdout_pred)
        fold_df = fold_stub_from_holdout(
            tmp_metrics,
            fit_time_sec=fit_time,
            infer_latency_ms=infer_ms,
            model_size_mb=pickle_size_mb(model_obj),
        )

    metrics, _ = add_model_result(
        model_name=model_name,
        library_source=model_name.split("::")[1].lower(),
        y_true=holdout_series.to_numpy(),
        y_pred=holdout_pred,
        timestamps=holdout_series.index,
        fold_metrics_df=fold_df,
        interpretability_note=baseline_notes[model_name],
        model_object=model_obj,
    )
    baseline_summary.append({"model_name": model_name, **metrics})

baseline_summary_df = pd.DataFrame(baseline_summary).sort_values("sMAPE")
display(baseline_summary_df)

## 6) LazyPredict Discovery Lab

Purpose:
- Rapidly scan model families on a **time-aware** holdout matrix.
- Build a ranked benchmark table.
- Identify candidate families for deeper manual engineering.

In [ ]:
lazy_discovery_df, lazy_prediction_matrix = run_lazypredict_discovery(
    X_train_one,
    y_train_one,
    X_eval_one,
    y_eval_one,
)

lazy_discovery_path = TABLE_DIR / "lazypredict_discovery.csv"
lazy_discovery_df.to_csv(lazy_discovery_path, index=False)

print("LazyPredict benchmark rows:", len(lazy_discovery_df))
print("Saved:", lazy_discovery_path)
display(lazy_discovery_df.head(20))

plt.figure(figsize=(12, 5))
plot_df = lazy_discovery_df.head(12).copy()
sns.barplot(data=plot_df, x="holdout_sMAPE", y="model_family", color="#1f77b4")
plt.title("LazyPredict Discovery Ranking (Top 12 by sMAPE)")
plt.xlabel("holdout sMAPE (lower is better)")
plt.ylabel("model family")
plt.show()

## 7) Selection of Top 3 Eligible Models

Rule enforced in this project:
- Manual Engineering Lab may use **only** top 3 eligible model families discovered here.
- No random model picks.

In [ ]:
eligible_lazy_df, lazy_eligibility_audit = select_eligible_lazypredict_models(
    lazy_discovery_df,
    top_k=8,
    max_fit_time_sec=120.0,
)

def manual_supported(model_family: str) -> bool:
    try:
        _ = build_manual_estimator(model_family, random_state=42)
        return True
    except Exception:
        return False

lazy_eligibility_audit = lazy_eligibility_audit.copy()
lazy_eligibility_audit["manual_supported"] = lazy_eligibility_audit["model_family"].map(manual_supported)

selected_top3 = (
    lazy_eligibility_audit[
        (lazy_eligibility_audit["is_eligible"]) & (lazy_eligibility_audit["manual_supported"])
    ]
    .sort_values(["holdout_sMAPE", "holdout_MAE", "holdout_RMSE"])
    .head(3)
    .reset_index(drop=True)
)

if len(selected_top3) < 3:
    fallback = (
        lazy_eligibility_audit[lazy_eligibility_audit["manual_supported"]]
        .sort_values(["holdout_sMAPE", "holdout_MAE", "holdout_RMSE"])
        .head(3)
        .reset_index(drop=True)
    )
    selected_top3 = fallback

selected_families = selected_top3["model_family"].tolist()

lazy_eligibility_path = TABLE_DIR / "lazypredict_eligibility_audit.csv"
selected_top3_path = TABLE_DIR / "selected_top3_families.csv"
lazy_eligibility_audit.to_csv(lazy_eligibility_path, index=False)
selected_top3.to_csv(selected_top3_path, index=False)

print("Selected top 3 families for manual engineering:")
for i, fam in enumerate(selected_families, start=1):
    print(f"{i}. {fam}")

print("Saved eligibility audit:", lazy_eligibility_path)
print("Saved top3 selection:", selected_top3_path)
display(selected_top3)

## 8) Manual Engineering Lab

Manual implementation follows the top-3 rule above.
Each candidate gets:
- explicit training logic
- recursive holdout forecasting
- rolling-origin backtest
- peak-demand threshold error analysis
- saved inference artifact

In [ ]:
manual_summary_rows = []
manual_hourly_error_frames = []

for family in selected_families:
    model_name = f"Manual::{family}"
    print(f"Running manual track for: {model_name}")

    def _factory(family_name=family):
        return build_manual_estimator(family_name, random_state=42)

    forecast_fn = make_recursive_forecast_fn(
        _factory,
        lags=LAGS,
        rolling_windows=ROLLING_WINDOWS,
        holiday_country=HOLIDAY_COUNTRY,
    )

    point_df, fold_df = run_rolling_backtest(train_series, model_name, forecast_fn, split_cfg)

    model = _factory()
    fit_start = time.perf_counter()
    model.fit(X_train_lag, y_train_lag)
    fit_time = time.perf_counter() - fit_start
    y_pred_holdout, infer_ms = recursive_lag_forecast(
        model=model,
        history=train_series,
        horizon=len(holdout_series),
        test_index=holdout_series.index,
        lags=LAGS,
        rolling_windows=ROLLING_WINDOWS,
        holiday_country=HOLIDAY_COUNTRY,
        feature_order=list(X_train_lag.columns),
    )

    hold_metrics = metric_bundle(holdout_series.to_numpy(), y_pred_holdout)
    if fold_df.empty:
        fold_df = fold_stub_from_holdout(
            hold_metrics,
            fit_time_sec=fit_time,
            infer_latency_ms=infer_ms,
            model_size_mb=pickle_size_mb(model),
        )

    peak_threshold = float(np.quantile(holdout_series.to_numpy(), 0.90))
    actual_peak = holdout_series.to_numpy() >= peak_threshold
    pred_peak = y_pred_holdout >= peak_threshold
    peak_recall = float(pred_peak[actual_peak].mean()) if actual_peak.any() else np.nan
    peak_precision = float(actual_peak[pred_peak].mean()) if pred_peak.any() else np.nan

    _, row = add_model_result(
        model_name=model_name,
        library_source="manual-engineering",
        y_true=holdout_series.to_numpy(),
        y_pred=y_pred_holdout,
        timestamps=holdout_series.index,
        fold_metrics_df=fold_df,
        interpretability_note=f"Manual implementation of LazyPredict-selected {family}.",
        model_object=model,
    )

    manual_model_path = MODEL_DIR / f"manual_{family.lower()}.pkl"
    joblib.dump(model, manual_model_path)
    saved_model_registry[model_name] = str(manual_model_path)

    manual_summary_rows.append(
        {
            "model_name": model_name,
            "family": family,
            "holdout_sMAPE": hold_metrics["sMAPE"],
            "holdout_MAE": hold_metrics["MAE"],
            "holdout_RMSE": hold_metrics["RMSE"],
            "calibration_metric": hold_metrics["calibration_metric"],
            "peak_recall": peak_recall,
            "peak_precision": peak_precision,
            "train_time_sec": fit_time,
            "infer_latency_ms": infer_ms,
            "artifact": str(manual_model_path),
        }
    )

    err_df = pd.DataFrame(
        {
            "timestamp": holdout_series.index,
            "model_name": model_name,
            "abs_error": np.abs(holdout_series.to_numpy() - y_pred_holdout),
        }
    )
    err_df["hour"] = err_df["timestamp"].dt.hour
    manual_hourly_error_frames.append(err_df)

    def _seed_eval(seed: int, family_name=family):
        m = build_manual_estimator(family_name, random_state=seed)
        m.fit(X_train_lag, y_train_lag)
        preds, _ = recursive_lag_forecast(
            model=m,
            history=train_series,
            horizon=len(holdout_series),
            test_index=holdout_series.index,
            lags=LAGS,
            rolling_windows=ROLLING_WINDOWS,
            holiday_country=HOLIDAY_COUNTRY,
            feature_order=list(X_train_lag.columns),
        )
        return metric_bundle(holdout_series.to_numpy(), preds)

    seed_eval_registry[model_name] = _seed_eval

manual_summary_df = pd.DataFrame(manual_summary_rows).sort_values("holdout_sMAPE")
manual_summary_df["risk_flag"] = np.where(
    (manual_summary_df["holdout_sMAPE"] > 25.0) | (manual_summary_df["peak_recall"] < 0.60),
    "watch",
    "ok",
)

display(manual_summary_df)

if manual_hourly_error_frames:
    hourly_errors = (
        pd.concat(manual_hourly_error_frames, ignore_index=True)
        .groupby(["model_name", "hour"], as_index=False)["abs_error"]
        .mean()
    )
    heat = hourly_errors.pivot(index="model_name", columns="hour", values="abs_error")
    plt.figure(figsize=(14, 4))
    sns.heatmap(heat, cmap="YlOrRd")
    plt.title("Manual Lab: Mean Absolute Error by Hour of Day")
    plt.show()

## 9) FLAML Optimization Lab

FLAML is treated as a full optimization track:
- explicit `time_budget`
- custom optimization metric = project primary metric (**sMAPE**)
- time-aware CV inside FLAML search
- holdout recursive forecast for final comparison

In [ ]:
flaml_result = run_flaml_optimization(
    X_train=X_train_lag,
    y_train=y_train_lag,
    time_budget_sec=FLAML_TIME_BUDGET_SEC,
    seed=42,
    estimator_list=["lgbm", "xgboost", "xgb_limitdepth", "rf", "extra_tree"],
    n_splits=3,
)

flaml_model_name = flaml_result["model_name"]
flaml_model = flaml_result["model_object"]
flaml_pred_holdout, flaml_infer_ms = recursive_lag_forecast(
    model=flaml_model,
    history=train_series,
    horizon=len(holdout_series),
    test_index=holdout_series.index,
    lags=LAGS,
    rolling_windows=ROLLING_WINDOWS,
    holiday_country=HOLIDAY_COUNTRY,
    feature_order=list(X_train_lag.columns),
)

flaml_holdout_metrics = metric_bundle(holdout_series.to_numpy(), flaml_pred_holdout)
flaml_metric_log = (flaml_result["automl"].best_result or {}).get("metric_for_logging", {})

flaml_fold_df = pd.DataFrame(
    [
        {
            "sMAPE": float(flaml_result.get("best_loss", np.nan)),
            "MAE": float(flaml_metric_log.get("val_mae", np.nan)),
            "RMSE": float(flaml_metric_log.get("val_rmse", np.nan)),
            "calibration_metric": np.nan,
            "fit_time_sec": float(flaml_result.get("fit_time_sec", np.nan)),
            "infer_latency_ms": float(flaml_infer_ms),
            "model_size_mb": pickle_size_mb(flaml_model),
        }
    ]
)

_, _ = add_model_result(
    model_name=flaml_model_name,
    library_source="flaml",
    y_true=holdout_series.to_numpy(),
    y_pred=flaml_pred_holdout,
    timestamps=holdout_series.index,
    fold_metrics_df=flaml_fold_df,
    interpretability_note="AutoML-selected boosted/tree model; moderate interpretability.",
    model_object=flaml_model,
)

flaml_model_path = MODEL_DIR / "flaml_best_model.pkl"
joblib.dump(flaml_model, flaml_model_path)
saved_model_registry[flaml_model_name] = str(flaml_model_path)

searched_estimators = [
    k for k, v in flaml_result["best_config_per_estimator"].items() if v is not None
]
flaml_history_rows = []
for it, payload in flaml_result["config_history"].items():
    estimator_name, cfg, tsec = payload
    flaml_history_rows.append(
        {
            "iteration": int(it),
            "estimator": estimator_name,
            "time_from_start_sec": float(tsec),
            "config": json.dumps(cfg),
        }
    )
flaml_history_df = pd.DataFrame(flaml_history_rows).sort_values("iteration")

print("FLAML searched estimators:", searched_estimators)
print("FLAML best estimator:", flaml_result["best_estimator"])
print("FLAML best config:")
print(flaml_result["best_config"])

display(flaml_history_df.head(20))
display(pd.DataFrame([flaml_holdout_metrics]))

if "manual_summary_df" in globals() and len(manual_summary_df) > 0:
    manual_best_smape = float(manual_summary_df["holdout_sMAPE"].min())
    delta = flaml_holdout_metrics["sMAPE"] - manual_best_smape
    print(f"FLAML vs best manual sMAPE delta: {delta:.4f} (negative means FLAML better)")

def _flaml_seed_eval(seed: int):
    out = run_flaml_optimization(
        X_train=X_train_lag,
        y_train=y_train_lag,
        time_budget_sec=FLAML_SEED_STABILITY_BUDGET_SEC,
        seed=seed,
        estimator_list=searched_estimators if searched_estimators else None,
        n_splits=2,
    )
    model = out["model_object"]
    preds, _ = recursive_lag_forecast(
        model=model,
        history=train_series,
        horizon=len(holdout_series),
        test_index=holdout_series.index,
        lags=LAGS,
        rolling_windows=ROLLING_WINDOWS,
        holiday_country=HOLIDAY_COUNTRY,
        feature_order=list(X_train_lag.columns),
    )
    return metric_bundle(holdout_series.to_numpy(), preds)

seed_eval_registry[flaml_model_name] = _flaml_seed_eval

## 10) PyCaret Experiment Lab

PyCaret is used as a full orchestration track:
- compare + tune + (blend when justified) + finalize + save for regression
- dedicated time-series experiment in parallel
- holdout recursive scoring for fair comparison with other tracks

Note: In PyCaret 4, `setup()` is integrated into OOP `Experiment(...).fit(...)`.

In [ ]:
pycaret_train_df = X_train_lag.copy()
pycaret_train_df["target"] = y_train_lag.values

pycaret_reg = run_pycaret_regression_workflow(
    train_df=pycaret_train_df,
    eval_df=X_eval_one.copy(),
    target_col="target",
    session_id=42,
    compare_include=["lr", "rf", "et"],
    tune_iter=4,
    save_path=str(MODEL_DIR / "pycaret_regression_final"),
    blend_top_n=2,
)

print("PyCaret regression selected label:", pycaret_reg["selected_label"])
print("PyCaret regression artifact:", pycaret_reg["saved_path"])
display(pycaret_reg["compare_leaderboard"].head(10))

pycaret_reg_model = clone(pycaret_reg["final_result"].pipeline)
reg_fit_start = time.perf_counter()
pycaret_reg_model.fit(X_train_lag, y_train_lag)
reg_fit_time = time.perf_counter() - reg_fit_start

pycaret_reg_pred_holdout, pycaret_reg_infer_ms = recursive_lag_forecast(
    model=pycaret_reg_model,
    history=train_series,
    horizon=len(holdout_series),
    test_index=holdout_series.index,
    lags=LAGS,
    rolling_windows=ROLLING_WINDOWS,
    holiday_country=HOLIDAY_COUNTRY,
    feature_order=list(X_train_lag.columns),
)

pycaret_reg_forecast_fn = make_recursive_forecast_fn(
    estimator_factory=lambda base=pycaret_reg["final_result"].pipeline: clone(base),
    lags=LAGS,
    rolling_windows=ROLLING_WINDOWS,
    holiday_country=HOLIDAY_COUNTRY,
)
pycaret_reg_points, pycaret_reg_fold = run_rolling_backtest(
    train_series,
    "PyCaretReg::Finalized",
    pycaret_reg_forecast_fn,
    split_cfg,
)

if pycaret_reg_fold.empty:
    pycaret_reg_fold = fold_stub_from_holdout(
        metric_bundle(holdout_series.to_numpy(), pycaret_reg_pred_holdout),
        fit_time_sec=reg_fit_time,
        infer_latency_ms=pycaret_reg_infer_ms,
        model_size_mb=pickle_size_mb(pycaret_reg_model),
    )

_, _ = add_model_result(
    model_name="PyCaretReg::Finalized",
    library_source="pycaret-regression",
    y_true=holdout_series.to_numpy(),
    y_pred=pycaret_reg_pred_holdout,
    timestamps=holdout_series.index,
    fold_metrics_df=pycaret_reg_fold,
    interpretability_note="PyCaret orchestrated compare/tune/finalize workflow on lag features.",
    model_object=pycaret_reg_model,
)
saved_model_registry["PyCaretReg::Finalized"] = pycaret_reg["saved_path"]

pycaret_ts = run_pycaret_timeseries_workflow(
    train_series=train_series,
    horizon=len(holdout_series),
    session_id=42,
    compare_include=["naive", "arima"],
    tune_iter=2,
    seasonal_period=24,
    save_path=str(MODEL_DIR / "pycaret_timeseries_final"),
)

pycaret_ts_name = pycaret_ts["model_name"]
pycaret_ts_pred = np.asarray(pycaret_ts["y_pred"], dtype=float)[: len(holdout_series)]

ts_metric_df = pycaret_ts["tune_result"].metrics.copy()
ts_fold_rows = []
for idx_name, row in ts_metric_df.iterrows():
    idx_str = str(idx_name).lower()
    if not idx_str.startswith("fold"):
        continue
    smape_val = row.get("SMAPE", np.nan)
    if pd.isna(smape_val):
        smape_val = row.get("MAPE", np.nan)
    ts_fold_rows.append(
        {
            "sMAPE": float(smape_val) if pd.notna(smape_val) else np.nan,
            "MAE": float(row.get("MAE", np.nan)),
            "RMSE": float(row.get("RMSE", np.nan)),
            "calibration_metric": np.nan,
            "fit_time_sec": np.nan,
            "infer_latency_ms": np.nan,
            "model_size_mb": pickle_size_mb(pycaret_ts["model_object"]),
        }
    )
pycaret_ts_fold = pd.DataFrame(ts_fold_rows)
if pycaret_ts_fold.empty:
    pycaret_ts_fold = fold_stub_from_holdout(
        metric_bundle(holdout_series.to_numpy(), pycaret_ts_pred),
        fit_time_sec=np.nan,
        infer_latency_ms=pycaret_ts.get("infer_latency_ms", np.nan),
        model_size_mb=pickle_size_mb(pycaret_ts["model_object"]),
    )

_, _ = add_model_result(
    model_name=pycaret_ts_name,
    library_source="pycaret-timeseries",
    y_true=holdout_series.to_numpy(),
    y_pred=pycaret_ts_pred,
    timestamps=holdout_series.index,
    fold_metrics_df=pycaret_ts_fold,
    interpretability_note="PyCaret time-series experiment on native forecasting models.",
    model_object=pycaret_ts["model_object"],
)
saved_model_registry[pycaret_ts_name] = pycaret_ts["saved_path"]

pycaret_track_compare = pd.DataFrame(
    [
        {
            "candidate": "PyCaretReg::Finalized",
            "holdout_sMAPE": metric_bundle(holdout_series.to_numpy(), pycaret_reg_pred_holdout)["sMAPE"],
        },
        {
            "candidate": pycaret_ts_name,
            "holdout_sMAPE": metric_bundle(holdout_series.to_numpy(), pycaret_ts_pred)["sMAPE"],
        },
    ]
).sort_values("holdout_sMAPE")
display(pycaret_track_compare)

def _pycaret_reg_seed_eval(seed: int):
    wf = run_pycaret_regression_workflow(
        train_df=pycaret_train_df,
        eval_df=X_eval_one.copy(),
        target_col="target",
        session_id=seed,
        compare_include=["lr"],
        tune_iter=1,
        save_path=None,
        blend_top_n=2,
    )
    m = clone(wf["final_result"].pipeline)
    m.fit(X_train_lag, y_train_lag)
    preds, _ = recursive_lag_forecast(
        model=m,
        history=train_series,
        horizon=len(holdout_series),
        test_index=holdout_series.index,
        lags=LAGS,
        rolling_windows=ROLLING_WINDOWS,
        holiday_country=HOLIDAY_COUNTRY,
        feature_order=list(X_train_lag.columns),
    )
    return metric_bundle(holdout_series.to_numpy(), preds)

seed_eval_registry["PyCaretReg::Finalized"] = _pycaret_reg_seed_eval

## 11) Unified Leaderboard and Final Model Ranking

Leaderboard includes:
- Top LazyPredict discovery rows
- Manual top-3 engineered models
- Best FLAML model
- Best PyCaret finalized models
- Baseline models

In [ ]:
lazy_for_leaderboard = lazy_discovery_df.head(5).copy()
for _, rec in lazy_for_leaderboard.iterrows():
    fam = rec["model_family"]
    y_pred = lazy_prediction_matrix[fam].to_numpy(dtype=float)
    lazy_metrics = {
        "sMAPE": float(rec["holdout_sMAPE"]),
        "MAE": float(rec["holdout_MAE"]),
        "RMSE": float(rec["holdout_RMSE"]),
        "calibration_metric": float(rec["calibration_metric"]),
    }
    fold_df = fold_stub_from_holdout(
        lazy_metrics,
        fit_time_sec=float(rec.get("fit_time_sec", np.nan)),
        infer_latency_ms=np.nan,
        model_size_mb=np.nan,
    )
    _ = add_model_result(
        model_name=f"LazyPredict::{fam}",
        library_source="lazypredict",
        y_true=y_eval_one.to_numpy(),
        y_pred=y_pred,
        timestamps=y_eval_one.index,
        fold_metrics_df=fold_df,
        interpretability_note="Discovery-stage benchmark; useful for family selection, not final deployment.",
        model_object=None,
    )

leaderboard_df = pd.DataFrame(leaderboard_rows)
leaderboard_df = finalize_leaderboard(leaderboard_df)

leaderboard_columns = [
    "project_name",
    "task_type",
    "library_source",
    "model_name",
    "cv_metric_mean",
    "cv_metric_std",
    "holdout_primary_metric",
    "holdout_secondary_metric",
    "holdout_tertiary_metric",
    "calibration_metric",
    "train_time_sec",
    "infer_latency_ms",
    "model_size_mb",
    "interpretability_note",
    "rank_score",
    "final_rank",
]
leaderboard_out = leaderboard_df[leaderboard_columns].copy()
leaderboard_path = ARTIFACT_DIR / "leaderboard_forecasting.csv"
leaderboard_out.to_csv(leaderboard_path, index=False)

all_points_df = pd.concat(holdout_point_frames, ignore_index=True)
horizon_metrics_df = horizon_band_metrics(all_points_df, short_max=24, medium_max=72)
horizon_rank_df = horizon_rank_table(horizon_metrics_df)

horizon_metrics_path = ARTIFACT_DIR / "horizon_metrics.csv"
horizon_rank_path = ARTIFACT_DIR / "horizon_rankings.csv"
horizon_metrics_df.to_csv(horizon_metrics_path, index=False)
horizon_rank_df.to_csv(horizon_rank_path, index=False)

print("Saved leaderboard:", leaderboard_path)
print("Saved horizon metrics:", horizon_metrics_path)
print("Saved horizon rankings:", horizon_rank_path)
display(leaderboard_out.head(10))

SEED_LIST = [11, 42]
top3_final = leaderboard_out.sort_values("final_rank").head(3)["model_name"].tolist()
seed_rows = []

for model_name in top3_final:
    eval_fn = seed_eval_registry.get(model_name)
    if eval_fn is None:
        seed_rows.append(
            {
                "model_name": model_name,
                "seed": "not_applicable",
                "sMAPE": np.nan,
                "MAE": np.nan,
                "RMSE": np.nan,
                "calibration_metric": np.nan,
                "elapsed_sec": np.nan,
                "note": "seed rerun not relevant or unavailable",
            }
        )
        continue

    for seed in SEED_LIST:
        start = time.perf_counter()
        try:
            m = eval_fn(seed)
            elapsed = time.perf_counter() - start
            seed_rows.append(
                {
                    "model_name": model_name,
                    "seed": seed,
                    "sMAPE": float(m.get("sMAPE", np.nan)),
                    "MAE": float(m.get("MAE", np.nan)),
                    "RMSE": float(m.get("RMSE", np.nan)),
                    "calibration_metric": float(m.get("calibration_metric", np.nan)),
                    "elapsed_sec": elapsed,
                    "note": "ok",
                }
            )
        except Exception as exc:
            elapsed = time.perf_counter() - start
            seed_rows.append(
                {
                    "model_name": model_name,
                    "seed": seed,
                    "sMAPE": np.nan,
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "calibration_metric": np.nan,
                    "elapsed_sec": elapsed,
                    "note": f"failed: {exc}",
                }
            )

seed_stability_df = pd.DataFrame(seed_rows)
seed_stability_path = ARTIFACT_DIR / "top3_seed_stability.csv"
seed_stability_df.to_csv(seed_stability_path, index=False)

print("Saved seed stability table:", seed_stability_path)
display(seed_stability_df)

## 12) Business Recommendation

Decision rule:
- **Winner:** model with best final rank score.
- **Safer second-best:** model with stronger calibration/latency trade-off even if slightly less accurate.

In [ ]:
ranked = leaderboard_out.sort_values("final_rank").reset_index(drop=True)
winner = ranked.iloc[0]

safer_candidates = ranked.copy()
safer_candidates = safer_candidates.sort_values(
    ["calibration_metric", "infer_latency_ms", "holdout_primary_metric"],
    ascending=[True, True, True],
)
safer = safer_candidates.iloc[0]
if safer["model_name"] == winner["model_name"] and len(safer_candidates) > 1:
    safer = safer_candidates.iloc[1]

print("Best overall winner:")
print(
    winner[
        [
            "model_name",
            "library_source",
            "holdout_primary_metric",
            "holdout_secondary_metric",
            "holdout_tertiary_metric",
            "calibration_metric",
        ]
    ]
)

print("\nSafer second-best candidate:")
print(
    safer[
        [
            "model_name",
            "library_source",
            "holdout_primary_metric",
            "calibration_metric",
            "infer_latency_ms",
        ]
    ]
)

recommendation_text = pd.DataFrame(
    [
        {
            "recommendation": "Champion model",
            "model": winner["model_name"],
            "when_to_use": "Primary production forecast where minimum sMAPE is required",
        },
        {
            "recommendation": "Safety challenger",
            "model": safer["model_name"],
            "when_to_use": "When latency/calibration stability is prioritized over raw accuracy",
        },
    ]
)
display(recommendation_text)


## 13) Inference / Deployment Path

Exported artifacts can be loaded for batch forecasting.
The snippet below chooses the highest-ranked model with an available saved artifact.

In [ ]:
SUPPORTED_DEPLOY_PREFIXES = ("Manual::", "FLAML::", "PyCaretReg::")

deployable_choice = None
deployable_path = None
for _, row in ranked.iterrows():
    model_name = row["model_name"]
    if not model_name.startswith(SUPPORTED_DEPLOY_PREFIXES):
        continue
    path = saved_model_registry.get(model_name)
    if path and Path(path).exists():
        deployable_choice = model_name
        deployable_path = Path(path)
        break

print("Chosen deployable model:", deployable_choice)
print("Artifact path:", deployable_path)

def forecast_next_24h_from_artifact(model_path: Path, history: pd.Series) -> pd.Series:
    model = joblib.load(model_path)
    y_hat, _ = recursive_lag_forecast(
        model=model,
        history=history,
        horizon=24,
        test_index=None,
        lags=LAGS,
        rolling_windows=ROLLING_WINDOWS,
        holiday_country=HOLIDAY_COUNTRY,
        feature_order=list(X_train_lag.columns),
    )
    next_index = pd.date_range(
        start=history.index[-1] + pd.Timedelta(hours=1),
        periods=24,
        freq="h",
    )
    return pd.Series(y_hat, index=next_index, name="forecast_kw")

if deployable_path is not None:
    sample_forecast = forecast_next_24h_from_artifact(deployable_path, train_series)
    display(sample_forecast.head())
else:
    print("No deployable lag-feature model artifact available in registry.")


## 14) Monitoring / Drift / Retraining Plan

Monitoring plan links model quality, drift, and operational actions.

In [ ]:
monitoring_plan = pd.DataFrame(
    [
        {
            "component": "Quality monitor",
            "trigger": "7-day rolling sMAPE > 1.15 * training baseline",
            "action": "Alert forecaster owner and run challenger evaluation",
        },
        {
            "component": "Drift monitor",
            "trigger": "Population mean shift > 2 sigma vs trailing 30-day demand",
            "action": "Trigger drift diagnostic report and retrain pipeline",
        },
        {
            "component": "Peak-risk monitor",
            "trigger": "3 consecutive peak-hour misses above P90 absolute error",
            "action": "Increase reserve buffer and escalate to operations",
        },
        {
            "component": "Retraining cadence",
            "trigger": "Scheduled weekly + event-driven retrain",
            "action": "Champion/challenger update and artifact re-registration",
        },
    ]
)

monitoring_plan_path = ARTIFACT_DIR / "monitoring_plan.csv"
monitoring_plan.to_csv(monitoring_plan_path, index=False)
print("Saved monitoring plan:", monitoring_plan_path)
display(monitoring_plan)

## 15) Limitations and Next Steps

Limitations:
- LazyPredict leaderboard uses one-step holdout features; final deployment comparisons rely on stricter recursive holdout.
- FLAML and PyCaret searches may favor different trade-offs under limited time budgets.
- Household-level demand patterns can shift due to external factors not captured in this dataset.

Next steps:
- Extend with exogenous weather/holiday/calendar shocks.
- Add hierarchical aggregation (daily/weekly) and reconciliation.
- Integrate online drift monitors and automatic rollback policy.